# Single-model ONNX → TensorRT evaluation (no Docker)

Edit `CHECKPOINT`, then run the cells in order. The notebook loads one `.pt`, exports dynamic FP16 ONNX with Ultralytics/PyTorch, validates it with ONNX, creates an ONNX Runtime TensorRT session directly, and evaluates the same model on LEVIR-Ship, TinyPerson, and Varroa.

TensorRT engines are built lazily by `TensorrtExecutionProvider` and cached in the run directory. NMS IoU is `0.5`.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import importlib.util
import json
import os

import onnx
import onnxruntime as ort
import torch

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'onnx_tensorrt_rerun_20261029':
    REPO_ROOT = REPO_ROOT.parent

# Change only this line to test another checkpoint.
CHECKPOINT = REPO_ROOT / 'exports/baseline_onnx_matrix/levirship/no_mosaic/yolov8/seed_42/hf_cache/runs/levirship/yolov8/seed_42/weights/best.pt'
DATA_ROOT = Path(os.environ.get('DATA_ROOT', '/home/ait01/jetson_seed42_benchmark/data'))
DATASETS = ('levirship', 'tinyperson', 'varroa')
CONF, NMS_IOU, MAX_DET, BATCH_SIZE = 0.001, 0.5, 300, 4
RUN_ID = datetime.now(timezone.utc).strftime('single_model_%Y%m%dT%H%M%SZ')
RUN_ROOT = REPO_ROOT / 'DUY_DEPLOY/results/tensorrt/notebook_runs' / RUN_ID
ONNX_PATH = RUN_ROOT / 'model_fp16_dynamic.onnx'
CACHE_ROOT = RUN_ROOT / 'trt_engine_cache'

assert CHECKPOINT.is_file(), f'Missing checkpoint: {CHECKPOINT}'
assert torch.cuda.is_available(), 'CUDA is required for FP16 export and TensorRT'
assert 'TensorrtExecutionProvider' in ort.get_available_providers(), ort.get_available_providers()
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print(json.dumps({
    'checkpoint': str(CHECKPOINT),
    'data_root': str(DATA_ROOT),
    'run_root': str(RUN_ROOT),
    'cuda': torch.cuda.get_device_name(0),
    'onnxruntime': ort.__version__,
}, indent=2))

In [ ]:
from ultralytics import YOLO

size = 512 if 'levirship' in CHECKPOINT.parts else 640
model = YOLO(str(CHECKPOINT))
produced = Path(model.export(
    format='onnx', imgsz=size, half=True, dynamic=True, batch=1,
    device=0, opset=17, simplify=True,
))
produced_model = onnx.load(str(produced), load_external_data=False)
onnx.checker.check_model(produced_model)
produced_model = None
ONNX_PATH.write_bytes(produced.read_bytes())
print('ONNX export and checker passed:', ONNX_PATH)

In [ ]:
import sys

evaluate_path = REPO_ROOT / 'onnx_tensorrt_rerun_20261029/source/evaluate.py'
spec = importlib.util.spec_from_file_location('rerun_evaluate', evaluate_path)
evaluate = importlib.util.module_from_spec(spec)
sys.modules['rerun_evaluate'] = evaluate
spec.loader.exec_module(evaluate)

CACHE_ROOT.mkdir(parents=True, exist_ok=True)
session_options = ort.SessionOptions()
session_options.add_session_config_entry('session.disable_cpu_ep_fallback', '1')
provider_options = {
    'device_id': 0,
    'trt_fp16_enable': True,
    'trt_engine_cache_enable': True,
    'trt_engine_cache_path': str(CACHE_ROOT),
}
session = ort.InferenceSession(
    str(ONNX_PATH),
    sess_options=session_options,
    providers=[('TensorrtExecutionProvider', provider_options)],
)
assert session.get_providers()[0] == 'TensorrtExecutionProvider', session.get_providers()
print('TensorRT session:', session.get_providers())
print('Input:', session.get_inputs()[0].name, session.get_inputs()[0].shape, session.get_inputs()[0].type)

In [ ]:
results = []
for dataset in DATASETS:
    for split in ('val', 'test'):
        images, ground_truth = evaluate.load_coco(DATA_ROOT, dataset, split)
        predicted = evaluate.predictions(
            session, images, evaluate.image_size(dataset), BATCH_SIZE,
            CONF, NMS_IOU, MAX_DET,
        )
        metrics = evaluate.evaluate(images, ground_truth, predicted)
        row = {
            'model': str(CHECKPOINT),
            'onnx': str(ONNX_PATH),
            'dataset': dataset,
            'split': split,
            'images': len(images),
            'ground_truth_boxes': sum(map(len, ground_truth.values())),
            'provider': 'TensorrtExecutionProvider',
            'batch_size': BATCH_SIZE,
            'conf': CONF,
            'nms_iou': NMS_IOU,
            **metrics,
        }
        results.append(row)
        output = RUN_ROOT / 'metrics' / dataset / f'{split}.json'
        output.parent.mkdir(parents=True, exist_ok=True)
        output.write_text(json.dumps(row, indent=2) + '\n', encoding='utf-8')
        print(json.dumps(row, sort_keys=True))

manifest = {
    'checkpoint': str(CHECKPOINT),
    'onnx': str(ONNX_PATH),
    'run_root': str(RUN_ROOT),
    'data_root': str(DATA_ROOT),
    'datasets': list(DATASETS),
    'provider': 'TensorrtExecutionProvider',
    'nms_iou': NMS_IOU,
    'results': results,
}
(RUN_ROOT / 'pipeline_manifest.json').write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
print('Wrote:', RUN_ROOT / 'pipeline_manifest.json')